# Kịch bản 3. Colab T4 with RAG & FineTunne Qwen/Qwen3-VL-8B-Instruct (T4 16GGPU không đáp ) -> Qwen/Qwen2.5-7B-Instruct

có một số điểm quan trọng cần làm rõ trước khi sửa code:

Về môi trường chạy: Đoạn code bạn cung cấp sử dụng thư viện unsloth, bitsandbytes và xformers. Những thư viện này bắt buộc cần GPU NVIDIA (CUDA) để hoạt động. Bạn không thể chạy đoạn code huấn luyện (training) này trên laptop có VGA Intel HD 620 được. Để chạy code này, bạn cần sử dụng Google Colab (với GPU T4 miễn phí) hoặc các dịch vụ Cloud GPU.
Về Input Model: Bạn yêu cầu dùng model Qwen/Qwen3-VL-8B-Instruct-GGUF. Thực tế, chúng ta không thể "train" (fine-tune) trực tiếp trên file GGUF. Chúng ta phải tải model gốc (định dạng PyTorch/HuggingFace) về để train, sau đó mới xuất (export) ra file GGUF để dùng trên laptop của bạn.
Dưới đây là đoạn code đã được sửa đổi hoàn chỉnh để:

Sử dụng model Qwen/Qwen3-VL-8B-Instruct.
Tương thích để train trên Google Colab (GPU T4).
Export ra file GGUF để chạy mượt mà trên laptop Intel của bạn.

Đoạn Code Copy sang Google Drive (Đã cập nhật tên thư mục)

Đoạn code này tương tự nhưng tôi đã cập nhật đường dẫn nguồn và đích để khớp với tên model mới:

In [ ]:
# Xóa sạch các thư viện đang gây xung đột
!pip uninstall unsloth trl -y

# Cài đặt lại Unsloth phiên bản ổn định (từ PyPI thay vì Git để tránh lỗi chưa fix)
# Bản PyPI mới nhất thường đã hỗ trợ Qwen2.5
!pip install --upgrade --no-cache-dir "unsloth[colab-new]" 
!pip install --upgrade --no-cache-dir trl

# Cài đặt các thư viện phụ trợ khác
!pip install gguf
!pip install --no-deps "xformers<0.0.26" triton accelerate bitsandbytes
!pip install llama-cpp-python

print("✅ Cài đặt hoàn tất. Vui lòng thực hiện Bước 2 ngay bây giờ!")

**Bước 2: KHỞI ĐỘNG LẠI SESSION (BẮT BUỘC)**
Đây là bước quan trọng nhất. Nếu bạn không làm bước này, Colab vẫn sẽ dùng code cũ của thư viện trong bộ nhớ và lỗi sẽ vẫn tồn tại.

Nhìn lên menu trên cùng của Colab.
*Chọn Runtime (Chạy thời) > Restart session (Khởi động lại phiên).*
Chọn "Restart" trong bảng hiện ra.
**Bước 3: Chạy lại code Training**
Sau khi màn hình Colab khởi động lại (biến mất hết code cũ), hãy chạy toàn bộ đoạn code training dưới đây (tôi đã giữ lại các sửa lỗi về **SFTConfig** và dùng thư viện ổn định):

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 1. LOAD MODEL QWEN2.5
print("Đang tải model Qwen2.5-7B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 2. CẤU HÌNH LORA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)

# 3. LOAD DỮ LIỆU
print("Đang tải dữ liệu...")
try:
    dataset = load_dataset(
        "json",
        data_files=["k-ai_pbi-2026.jsonl"], 
        split="train"
    )

    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts }

    dataset = dataset.map(formatting_prompts_func, batched = True)
    print(f"✅ Dữ liệu: {len(dataset)} mẫu")
except Exception as e:
    print(f"Lỗi dữ liệu: {e}")

# 4. TRAINING
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,         # Sửa lỗi SFTConfig
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        packing = False,
    ),
)

print("🚀 Training...")
trainer.train()

# 5. LƯU & EXPORT
print("💾 Lưu LoRA...")
model.save_pretrained("iqw25_7bkai_pbi_lora")
tokenizer.save_pretrained("iqw25_7bkai_pbi_lora")

print("🔄 Export GGUF...")
model.save_pretrained_gguf("iqw25_7bkai_pbi_gguf", tokenizer, quantization_method = "q4_k_m")
print("✅ Hoàn tất!")

In [ ]:
from google.colab import drive
import os

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn
# Nguồn: Thư mục chứa file GGUF vừa tạo
source_dir = "/content/iqw25_7bkai_pbi_gguf_gguf"

# Đích: Thư mục trên Google Drive (Đã đổi tên để phân biệt với model cũ)
target_dir = "/content/drive/MyDrive/iqw25_7bkai_pbi"

# 3. Tạo thư mục đích nếu chưa có
os.makedirs(target_dir, exist_ok=True)

# 4. Thực hiện lệnh Copy
print(f"Đang bắt đầu copy từ {source_dir} sang Google Drive...")
print("⏳ Quá trình này có thể mất từ 10 đến 15 phút vì model 8B ~ 5.6GB.")

# Thực hiện lệnh copy
!cp -r {source_dir} {target_dir}

print("\n>>> HOÀN TẤT! Model iqw25_7bkai_pbi đã sẵn sàng trong thư mục iqw25_7bkai_pbi trên Google Drive.")